# Credit Scoring: Training, Evaluation and Fairness Audit

Classify loan applicants as good or bad credit risks on the German Credit dataset (OpenML `credit-g`).
Educational project: not a lending tool.

Run all cells top to bottom (Colab or Jupyter). It writes `credit_model.pkl`, `model_info.json`, `credit_features.py` and `requirements.txt` next to the notebook. Commit those with the repo.

## 1. Shared feature code

The engineered feature lives in its own module so the app can load the saved model.

In [ ]:
%%writefile credit_features.py
"""Feature engineering shared by the training notebook and the Streamlit app.

It lives in its own module so that the pickled model can be loaded anywhere this file is importable.
"""


def add_credit_per_month(X):
    """Add the engineered feature: monthly instalment size (credit amount spread over the duration)."""
    X = X.copy()
    X["credit_per_month"] = X["credit_amount"] / (X["duration"] + 1)
    return X

## 2. Setup

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
                             classification_report, make_scorer, recall_score, roc_auc_score)
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from credit_features import add_credit_per_month

SEED = 42
# Sensitive attributes: excluded from the model, but still used in the fairness audit below
DROP_COLUMNS = ["personal_status", "foreign_worker"]
# Cost matrix from the dataset documentation: approving a bad borrower costs 5x more than rejecting a good one
COST_BAD_APPROVED, COST_GOOD_REJECTED = 5, 1

## 3. Data

In [ ]:
CSV_PATH = Path("credit_g.csv")
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH, keep_default_na=False)   # keep the literal value "none" as text
else:
    from sklearn.datasets import fetch_openml
    df = fetch_openml("credit-g", version=1, as_frame=True).frame.copy()
    df.to_csv(CSV_PATH, index=False)   # snapshot, so the project no longer depends on the network
print(df.shape, "| missing values:", int(df.isna().sum().sum()))
df.head()

In [ ]:
y = (df["class"] == "good").astype(int)        # 1 = good credit, 0 = bad credit
X_all = df.drop(columns="class")
INPUT_COLS = [c for c in X_all.columns if c not in DROP_COLUMNS]

print(f"Good: {y.mean():.1%} | Bad: {1 - y.mean():.1%}")
print(f"Accuracy of approving everyone: {y.mean():.3f}   <- any model has to beat this")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["duration", "credit_amount"]):
    df.boxplot(column=col, by="class", ax=ax); ax.set_title(col); ax.set_xlabel("")
plt.suptitle(""); plt.tight_layout(); plt.show()

## 4. Split and column groups

Categorical columns are found by exclusion, so it does not matter whether the data arrives as `object`, `category` or `str`. The assertions guarantee that every input column reaches the model. (An earlier version selected `include='object'`, which matched nothing for OpenML `category` columns and silently dropped every categorical feature.)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, stratify=y, random_state=SEED)
X_tr, X_te = X_train[INPUT_COLS], X_test[INPUT_COLS]

raw_numeric = X_all[INPUT_COLS].select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in INPUT_COLS if c not in raw_numeric]
numeric_cols = raw_numeric + ["credit_per_month"]          # credit_per_month is created inside the pipeline

assert len(categorical_cols) > 0, "No categorical columns found: they would be silently dropped."
assert set(numeric_cols + categorical_cols) == set(INPUT_COLS) | {"credit_per_month"}
print(len(numeric_cols), "numeric features:", numeric_cols)
print(len(categorical_cols), "categorical features:", categorical_cols)

## 5. Pipeline and model comparison (cross-validation on the training split only)

In [ ]:
def make_pipe(classifier):
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ])
    return Pipeline([
        ("features", FunctionTransformer(add_credit_per_month)),
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

BASELINE = "Baseline (approve everyone)"
models = {
    BASELINE: DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Logistic Regression (balanced)": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED),
    "Random Forest (balanced)": RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                       class_weight="balanced_subsample", random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
scoring = {"roc_auc": "roc_auc", "accuracy": "accuracy", "recall_bad": make_scorer(recall_score, pos_label=0)}

rows = []
for name, clf in models.items():
    s = cross_validate(make_pipe(clf), X_tr, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"model": name, "ROC-AUC": s["test_roc_auc"].mean(), "ROC-AUC std": s["test_roc_auc"].std(),
                 "accuracy": s["test_accuracy"].mean(), "recall (bad loans)": s["test_recall_bad"].mean()})
cv_table = pd.DataFrame(rows).set_index("model")
display(cv_table.round(3))

best_name = cv_table.drop(index=BASELINE).sort_values("ROC-AUC", ascending=False).index[0]
print("Selected model (highest cross-validated ROC-AUC):", best_name)

## 6. Choose the decision threshold with the cost matrix

Approving a bad borrower costs 5x more than rejecting a good one, so 0.5 is the wrong cutoff. The threshold is chosen on out-of-fold predictions from the training split, and the test set is used only once, afterwards.

In [ ]:
def avg_cost(y_true, p_good, threshold):
    approve = p_good >= threshold
    bad_approved = int((approve & (y_true == 0)).sum())
    good_rejected = int((~approve & (y_true == 1)).sum())
    return (COST_BAD_APPROVED * bad_approved + COST_GOOD_REJECTED * good_rejected) / len(y_true)

oof = cross_val_predict(make_pipe(clone(models[best_name])), X_tr, y_train,
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                        method="predict_proba", n_jobs=-1)[:, 1]

thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
costs = [avg_cost(y_train.to_numpy(), oof, t) for t in thresholds]
best_t = float(thresholds[int(np.argmin(costs))])

plt.figure(figsize=(6, 4))
plt.plot(thresholds, costs); plt.axvline(best_t, color="r", ls="--", label=f"chosen threshold = {best_t:.2f}")
plt.xlabel("Approval threshold (probability of good)"); plt.ylabel("Average cost per applicant")
plt.title("Cost vs threshold (out-of-fold, training split)"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Held-out test results

In [ ]:
eval_pipe = make_pipe(clone(models[best_name])).fit(X_tr, y_train)
p_test = eval_pipe.predict_proba(X_te)[:, 1]
y_true = y_test.to_numpy()

def summarize(policy, t):
    pred = (p_test >= t).astype(int)
    return {"policy": policy, "threshold": t, "cost per applicant": avg_cost(y_true, p_test, t),
            "accuracy": accuracy_score(y_true, pred), "recall (bad loans)": recall_score(y_true, pred, pos_label=0),
            "approval rate": pred.mean()}

summary = pd.DataFrame([
    summarize("model, default threshold", 0.5),
    summarize("model, cost-tuned threshold", best_t),
    summarize("approve everyone", 0.0),
    summarize("reject everyone", 1.01),
]).set_index("policy").round(3)
display(summary)

test_auc = float(roc_auc_score(y_true, p_test))
pred_t = (p_test >= best_t).astype(int)
print(f"Test ROC-AUC: {test_auc:.4f}\n")
print(classification_report(y_true, pred_t, target_names=["bad", "good"], digits=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_true, pred_t, display_labels=["bad", "good"], ax=axes[0], colorbar=False)
axes[0].set_title(f"Confusion matrix (threshold {best_t:.2f})")
RocCurveDisplay.from_predictions(y_true, p_test, ax=axes[1]); axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].set_title("ROC curve (test set)")
plt.tight_layout(); plt.show()

## 8. What drives the predictions (permutation importance on the test set)

In [ ]:
perm = permutation_importance(eval_pipe, X_te, y_test, scoring="roc_auc", n_repeats=10, random_state=SEED)
importance = pd.Series(perm.importances_mean, index=INPUT_COLS).sort_values()
importance.tail(12).plot(kind="barh", figsize=(7, 5), title="Drop in ROC-AUC when a feature is shuffled")
plt.tight_layout(); plt.show()

## 9. Fairness audit

Groups are defined on the held-out set, including the sensitive attributes excluded from the model, because other features can act as proxies. Group sizes are small, so read the table as a direction, not as proof.

- **approval rate:** share predicted good
- **actual good rate:** share who really were good
- **bad loans caught:** share of truly bad applicants the model rejected

In [ ]:
groups = {}
if "age" in X_test:
    groups["age"] = {"under 25": X_test["age"].to_numpy() < 25, "25 and over": X_test["age"].to_numpy() >= 25}
for col in ["foreign_worker", "personal_status"]:
    if col in X_test:
        values = X_test[col].astype(str).to_numpy()
        groups[col] = {v: values == v for v in sorted(set(values))}

rows = []
for attribute, members in groups.items():
    for group, mask in members.items():
        n = int(mask.sum())
        if n == 0:
            continue
        bad = y_true[mask] == 0
        rows.append({"attribute": attribute, "group": group, "n": n,
                     "approval rate": round(float(pred_t[mask].mean()), 3),
                     "actual good rate": round(float(y_true[mask].mean()), 3),
                     "bad loans caught": round(float((pred_t[mask][bad] == 0).mean()), 3) if bad.any() else np.nan})
pd.DataFrame(rows).set_index(["attribute", "group"])

## 10. Save

The deployed model is refit on all 1,000 rows with the threshold chosen above. The test results above come from the model trained on the training split only.

In [ ]:
final_model = make_pipe(clone(models[best_name])).fit(X_all[INPUT_COLS], y)
joblib.dump(final_model, "credit_model.pkl")

numeric_info = {}
for c in raw_numeric:
    col = df[c]
    is_int = bool((col % 1 == 0).all())
    cast = int if is_int else float
    numeric_info[c] = {"min": cast(col.min()), "max": cast(col.max()), "median": cast(col.median()), "is_int": is_int}

model_info = {
    "inputs": INPUT_COLS,
    "numeric": numeric_info,
    "categorical": {c: sorted(df[c].astype(str).unique().tolist()) for c in categorical_cols},
    "threshold": best_t,
    "cost_bad_approved": COST_BAD_APPROVED,
    "cost_good_rejected": COST_GOOD_REJECTED,
    "model": best_name,
    "excluded_sensitive_columns": DROP_COLUMNS,
    "test_metrics": {
        "roc_auc": round(test_auc, 4),
        "accuracy": round(float(accuracy_score(y_true, pred_t)), 4),
        "recall_bad_loans": round(float(recall_score(y_true, pred_t, pos_label=0)), 4),
        "cost_per_applicant": round(float(avg_cost(y_true, p_test, best_t)), 4),
        "baseline_accuracy_approve_everyone": round(float(y_true.mean()), 4),
        "n_test": int(len(y_true)),
    },
    "n_rows_trained_on": int(len(df)),
    "scikit_learn_version": sklearn.__version__,
}
Path("model_info.json").write_text(json.dumps(model_info, indent=2))

# A .pkl only loads reliably on the scikit-learn version that saved it, so pin that version for the app
Path("requirements.txt").write_text("\n".join(["streamlit", f"scikit-learn=={sklearn.__version__}", "pandas", "numpy", "joblib"]) + "\n")

print("Saved credit_model.pkl, model_info.json and requirements.txt (scikit-learn", sklearn.__version__ + ")")
model_info["test_metrics"]